In [1]:
import numpy as np
import pandas as pd
import random
import os
import glob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import faiss
import torch
from sentence_transformers import SentenceTransformer

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используется устройство: {device}")

Используется устройство: cpu


In [3]:
kb_folder = "data/knowledge_base/"
file_paths = sorted(glob.glob(os.path.join(kb_folder, "*.txt")))
print(f"Найдено файлов: {len(file_paths)}")

knowledge_base = []
for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    # Первая строка — заголовок, остальное — текст
    lines = content.split('\n', 1)
    title = lines[0].strip()
    text = lines[1].strip() if len(lines) > 1 else ""
    doc_id = int(os.path.basename(file_path).split('_')[1].split('.')[0])
    knowledge_base.append({"id": doc_id, "title": title, "text": text})

print(f"Загружено документов: {len(knowledge_base)}")
print("\nПримеры документов (заголовки):")
for i, doc in enumerate(knowledge_base[:5]):
    print(f"{i+1}. {doc['title']}")

Найдено файлов: 20
Загружено документов: 20

Примеры документов (заголовки):
1. Эмбеддинги текстов
2. FAISS и поиск ближайших соседей
3. Чанкинг и overlap при разбиении текста
4. Косинусное сходство и его применение
5. Retrieval-Augmented Generation (RAG)


In [4]:
def chunk_text(text, chunk_size_words=150, overlap_words=30):
    words = text.split()
    chunks = []
    step = chunk_size_words - overlap_words
    if step <= 0:
        step = chunk_size_words
    for start in range(0, len(words), step):
        end = start + chunk_size_words
        chunk_words = words[start:end]
        if len(chunk_words) < 10:
            continue
        chunks.append(' '.join(chunk_words))
    return chunks

CHUNK_SIZE = 150
OVERLAP = 30

chunks = []
for doc in knowledge_base:
    doc_chunks = chunk_text(doc['text'], CHUNK_SIZE, OVERLAP)
    for idx, chunk_text_str in enumerate(doc_chunks):
        chunks.append({
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'chunk_index': idx,
            'text': chunk_text_str
        })

print(f"Получено чанков: {len(chunks)}")
print("\nПример чанка:")
print(chunks[0]['text'][:150], "...")

Получено чанков: 40

Пример чанка:
Эмбеддинги текстов — это плотные векторные представления, которые сохраняют семантический смысл слов, предложений или целых документов. Они получаются ...


In [5]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
model.to(device)

chunk_texts = [c['text'] for c in chunks]
embeddings = model.encode(chunk_texts, convert_to_numpy=True, show_progress_bar=True)
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"Размерность эмбеддингов: {dim}")
print(f"Количество векторов в индексе: {index.ntotal}")

def search(query, idx, chks, top_k=3):
    """Поиск с использованием переданных индекса и списка чанков"""
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = idx.search(q_emb, top_k)
    results = []
    for score, idx_i in zip(scores[0], indices[0]):
        results.append({'chunk': chks[idx_i], 'score': float(score)})
    return results

test_queries = ["Что такое эмбеддинги?", "Как работает FAISS?", "Что такое overlap?"]
for q in test_queries:
    print(f"\nЗапрос: {q}")
    for res in search(q, index, chunks, top_k=2):
        print(f"  - {res['chunk']['doc_title']} (score={res['score']:.3f})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Размерность эмбеддингов: 384
Количество векторов в индексе: 40

Запрос: Что такое эмбеддинги?
  - Эмбеддинги текстов (score=0.407)
  - Чанкинг и overlap при разбиении текста (score=0.362)

Запрос: Как работает FAISS?
  - Индексирование в FAISS: типы индексов (score=0.422)
  - Чанкинг и overlap при разбиении текста (score=0.370)

Запрос: Что такое overlap?
  - Косинусное сходство и его применение (score=0.482)
  - Чанкинг и overlap при разбиении текста (score=0.450)


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidv = TfidfVectorizer().fit(chunk_texts)
tfidf_matrix = tfidv.transform(chunk_texts)

def hybrid_search(query, idx, chks, tfidv, tfidf_matrix, top_k=3, alpha=0.7, dense_candidates_mult=10):
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    
    n_dense = max(top_k * dense_candidates_mult, 20)
    dense_scores, dense_indices = idx.search(q_emb, n_dense)
    
    q_tfidf = tfidv.transform([query])
    sparse_scores_all = cosine_similarity(q_tfidf, tfidf_matrix).flatten()
    
    candidate_indices = set(dense_indices[0])
    candidate_scores = {}
    for i in candidate_indices:
        pos = list(dense_indices[0]).index(i)
        dense_score = float(dense_scores[0][pos])
        sparse_score = float(sparse_scores_all[i])
        combined = alpha * dense_score + (1 - alpha) * sparse_score
        candidate_scores[i] = combined
    
    sorted_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    results = []
    for idx_i, score in sorted_candidates:
        results.append({'chunk': chks[idx_i], 'score': score})
    return results

def search(query, idx, chks, top_k=3):
    return hybrid_search(query, idx, chks, tfidv, tfidf_matrix, top_k=top_k, alpha=0.7)

In [24]:
def evaluate_retrieval(queries, idx, chks, tfidv, tfidf_matrix, k=3, alpha=0.5, dense_candidates_mult=1000):
    results = []
    for q in queries:
        query = q["query"]
        expected = q["expected_title"]
        
        retrieved = hybrid_search(query, idx, chks, tfidv, tfidf_matrix, top_k=k, alpha=alpha, dense_candidates_mult=dense_candidates_mult)
        retrieved_titles = [r['chunk']['doc_title'] for r in retrieved]
        
        hit = 0
        mrr = 0.0
        rank = -1
        for r, title in enumerate(retrieved_titles, start=1):
            if title == expected:
                hit = 1
                mrr = 1.0 / r
                rank = r
                break
        
        results.append({
            'query': query,
            'expected_source': expected,
            'retrieved_sources': ', '.join(retrieved_titles),
            'hit_at_k': hit,
            'recall_at_k': hit,
            'mrr_at_k': mrr,
            'rank_of_first_relevant': rank
        })
    return pd.DataFrame(results)

queries_eval = [
    {"query": "Как получить векторное представление текста?", "expected_title": "Эмбеддинги текстов"},
    {"query": "Библиотека для быстрого поиска ближайших соседей", "expected_title": "FAISS и поиск ближайших соседей"},
    {"query": "Зачем разбивать документы на фрагменты с перекрытием?", "expected_title": "Чанкинг и overlap при разбиении текста"},
    {"query": "Мера сходства, основанная на угле между векторами", "expected_title": "Косинусное сходство и его применение"},
    {"query": "Архитектура, объединяющая поиск и генерацию", "expected_title": "Retrieval-Augmented Generation (RAG)"},
    {"query": "Как оценить, насколько хорошо поиск находит нужный документ?", "expected_title": "Метрики оценки качества retrieval: hit, recall, MRR"},
    {"query": "Модель для русскоязычных эмбеддингов предложений", "expected_title": "Sentence‑transformers модели для русского языка"},
    {"query": "Классический метод векторизации, основанный на частоте терминов", "expected_title": "TF‑IDF как альтернатива эмбеддингам"},
    {"query": "Что происходит при добавлении новых документов в индекс?", "expected_title": "Обновление базы знаний и переиндексация"},
    {"query": "Упрощённая версия RAG для демонстрации механики", "expected_title": "Mini‑RAG пайплайн"}
]

k = 3
df_eval = evaluate_retrieval(queries_eval, index, chunks, tfidv, tfidf_matrix, k=k, alpha=0.5, dense_candidates_mult=1000)

print(f"Оценка retrieval (k={k}):")
print(df_eval[['query', 'expected_source', 'hit_at_k', 'mrr_at_k']].to_string(index=False))
print(f"\nСредний hit@{k}: {df_eval['hit_at_k'].mean():.3f}")
print(f"Средний recall@{k}: {df_eval['recall_at_k'].mean():.3f}")
print(f"Средний MRR@{k}: {df_eval['mrr_at_k'].mean():.3f}")

os.makedirs('artifacts', exist_ok=True)
df_eval[['query', 'expected_source', 'retrieved_sources', 'hit_at_k', 'rank_of_first_relevant']].to_csv(
    'artifacts/retrieval_eval.csv',
    index=False,
    encoding='utf-8-sig'
)
print("\nCSV-файл сохранён в artifacts/retrieval_eval.csv")

Оценка retrieval (k=3):
                                                          query                                     expected_source  hit_at_k  mrr_at_k
                   Как получить векторное представление текста?                                  Эмбеддинги текстов         1       1.0
               Библиотека для быстрого поиска ближайших соседей                     FAISS и поиск ближайших соседей         1       1.0
          Зачем разбивать документы на фрагменты с перекрытием?              Чанкинг и overlap при разбиении текста         1       1.0
              Мера сходства, основанная на угле между векторами                Косинусное сходство и его применение         1       1.0
                    Архитектура, объединяющая поиск и генерацию                Retrieval-Augmented Generation (RAG)         1       0.5
   Как оценить, насколько хорошо поиск находит нужный документ? Метрики оценки качества retrieval: hit, recall, MRR         1       1.0
               Модель дл

In [26]:
df_hit1 = evaluate_retrieval(queries_eval, index, chunks, tfidv, tfidf_matrix, k=1)
hit1 = df_hit1['hit_at_k'].mean()
print(f"Средний hit@1 = {hit1:.3f}")
print(f"Средний hit@3 = {df_eval['hit_at_k'].mean():.3f}")
print("Вывод: увеличение top_k с 1 до 3 повышает вероятность нахождения релевантного документа,")
print("что критически важно для RAG, так как генератор может использовать информацию из любого из top-k чанков.")

Средний hit@1 = 0.800
Средний hit@3 = 1.000
Вывод: увеличение top_k с 1 до 3 повышает вероятность нахождения релевантного документа,
что критически важно для RAG, так как генератор может использовать информацию из любого из top-k чанков.


In [27]:
new_docs = [
    {"id": 21, "title": "FAISS и поиск ближайших соседей", 
     "text": "FAISS (Facebook AI Similarity Search) — это библиотека для эффективного поиска ближайших соседей и кластеризации векторных представлений. Она реализует алгоритмы, такие как IVF, HNSW, PQ, и позволяет выполнять поиск на миллиардах векторов за миллисекунды. FAISS особенно полезна для задач поиска семантически похожих документов, изображений и рекомендательных систем."},
    {"id": 22, "title": "Retrieval-Augmented Generation (RAG)", 
     "text": "Retrieval-Augmented Generation (RAG) — это архитектура, объединяющая поиск информации (retrieval) и генерацию текста (generation). Сначала система находит релевантные документы или чанки из базы знаний, а затем передаёт их в языковую модель (LLM) для формирования ответа. RAG позволяет использовать актуальные внешние данные и уменьшает галлюцинации LLM."},
    {"id": 23, "title": "Метрики оценки качества retrieval: hit, recall, MRR", 
     "text": "Для оценки качества поиска используются метрики: hit@k (наличие хотя бы одного релевантного документа в top‑k), recall@k (доля найденных релевантных документов) и MRR (Mean Reciprocal Rank) — среднее обратное ранжирование первого релевантного документа. Эти метрики позволяют объективно сравнивать разные стратегии поиска."},
    {"id": 24, "title": "TF‑IDF как альтернатива эмбеддингам", 
     "text": "TF‑IDF (Term Frequency–Inverse Document Frequency) — это классический метод векторизации текста, основанный на частоте терминов. В отличие от плотных эмбеддингов, TF‑IDF создаёт разреженные векторы, где важность слова тем выше, чем чаще оно встречается в документе и реже — во всей коллекции. TF‑IDF хорошо работает для точного совпадения ключевых слов и часто используется в гибридном поиске."},
    {"id": 25, "title": "Практический RAG с использованием FAISS", 
     "text": "В этом разделе описывается, как объединить FAISS для быстрого поиска и генерацию ответов. RAG (Retrieval-Augmented Generation) становится ещё эффективнее, когда индекс FAISS оптимизирован. Применение FAISS в RAG позволяет масштабировать поиск до миллионов документов."}
]

print("ДО обновления базы знаний:")
compare_queries = [
    ("Библиотека для быстрого поиска ближайших соседей", "FAISS и поиск ближайших соседей"),
    ("Архитектура, объединяющая поиск и генерацию", "Retrieval-Augmented Generation (RAG)"),
    ("Как оценить, насколько хорошо поиск находит нужный документ?", "Метрики оценки качества retrieval: hit, recall, MRR"),
    ("Классический метод векторизации, основанный на частоте терминов", "TF‑IDF как альтернатива эмбеддингам"),
    ("RAG с использованием FAISS", "Практический RAG с использованием FAISS")
]

for q, exp in compare_queries:
    res = search(q, index, chunks, top_k=3)
    titles = [r['chunk']['doc_title'] for r in res]
    print(f"Запрос: '{q}' -> найдены заголовки: {titles}")

updated_kb = knowledge_base + new_docs
updated_chunks = []
for doc in updated_kb:
    doc_chunks = chunk_text(doc['text'], CHUNK_SIZE, OVERLAP)
    for idx, chunk_text_str in enumerate(doc_chunks):
        updated_chunks.append({
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'chunk_index': idx,
            'text': chunk_text_str
        })

updated_texts = [c['text'] for c in updated_chunks]
updated_embeddings = model.encode(updated_texts, convert_to_numpy=True, show_progress_bar=True)
faiss.normalize_L2(updated_embeddings)
updated_index = faiss.IndexFlatIP(dim)
updated_index.add(updated_embeddings)

updated_tfidv = TfidfVectorizer().fit(updated_texts)
updated_tfidf_matrix = updated_tfidv.transform(updated_texts)

def search_updated(query, top_k=3):
    return hybrid_search(query, updated_index, updated_chunks, updated_tfidv, updated_tfidf_matrix, top_k=top_k, alpha=0.7)

print("\nПОСЛЕ обновления базы знаний:")
comparison_data = []
for q, exp in compare_queries:
    before = search(q, index, chunks, top_k=3)
    after = search_updated(q, top_k=3)
    before_titles = [r['chunk']['doc_title'] for r in before]
    after_titles = [r['chunk']['doc_title'] for r in after]
    print(f"Запрос: '{q}'")
    print(f"  До: {before_titles}")
    print(f"  После: {after_titles}")
    comparison_data.append({
        'query': q,
        'before_retrieved_sources': ', '.join(before_titles),
        'after_retrieved_sources': ', '.join(after_titles),
        'changed': (exp in after_titles) and (exp not in before_titles)
    })

df_compare = pd.DataFrame(comparison_data)
df_compare.to_csv('artifacts/retrieval_before_after_update.csv', index=False, encoding='utf-8-sig')
print("\nСравнение сохранено в artifacts/retrieval_before_after_update.csv")

ДО обновления базы знаний:
Запрос: 'Библиотека для быстрого поиска ближайших соседей' -> найдены заголовки: ['FAISS и поиск ближайших соседей', 'Гибридный поиск в RAG', 'Метрики оценки качества retrieval: hit, recall, MRR']
Запрос: 'Архитектура, объединяющая поиск и генерацию' -> найдены заголовки: ['Гибридный поиск в RAG', 'Ранжирование в поиске и реранжирование', 'Будущее RAG и тенденции развития']
Запрос: 'Как оценить, насколько хорошо поиск находит нужный документ?' -> найдены заголовки: ['Оценка retrieval с использованием бенчмарка', 'Метрики оценки качества retrieval: hit, recall, MRR', 'Параметры чанкинга и их влияние на метрики']
Запрос: 'Классический метод векторизации, основанный на частоте терминов' -> найдены заголовки: ['TF‑IDF как альтернатива эмбеддингам', 'Векторизация текстов: практические аспекты', 'Эмбеддинги текстов']
Запрос: 'RAG с использованием FAISS' -> найдены заголовки: ['Будущее RAG и тенденции развития', 'Retrieval-Augmented Generation (RAG)', 'Чанкинг и ove

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


ПОСЛЕ обновления базы знаний:
Запрос: 'Библиотека для быстрого поиска ближайших соседей'
  До: ['FAISS и поиск ближайших соседей', 'Гибридный поиск в RAG', 'Метрики оценки качества retrieval: hit, recall, MRR']
  После: ['FAISS и поиск ближайших соседей', 'Гибридный поиск в RAG', 'FAISS и поиск ближайших соседей']
Запрос: 'Архитектура, объединяющая поиск и генерацию'
  До: ['Гибридный поиск в RAG', 'Ранжирование в поиске и реранжирование', 'Будущее RAG и тенденции развития']
  После: ['Retrieval-Augmented Generation (RAG)', 'Гибридный поиск в RAG', 'Ранжирование в поиске и реранжирование']
Запрос: 'Как оценить, насколько хорошо поиск находит нужный документ?'
  До: ['Оценка retrieval с использованием бенчмарка', 'Метрики оценки качества retrieval: hit, recall, MRR', 'Параметры чанкинга и их влияние на метрики']
  После: ['Оценка retrieval с использованием бенчмарка', 'Метрики оценки качества retrieval: hit, recall, MRR', 'Параметры чанкинга и их влияние на метрики']
Запрос: 'Классичес

In [28]:
def extractive_answer(query, top_k=3, alpha=0.7):
    retrieved = hybrid_search(query, updated_index, updated_chunks, 
                              updated_tfidv, updated_tfidf_matrix, 
                              top_k=top_k, alpha=alpha)
    sources = list(set([r['chunk']['doc_title'] for r in retrieved]))
    context = " ".join([r['chunk']['text'] for r in retrieved])
    
    sentences = [s.strip() for s in context.split('.') if len(s.strip()) > 10]
    if not sentences:
        return "Не удалось сформировать ответ (нет достаточного контекста).", sources
    
    query_vec = updated_tfidv.transform([query])
    sent_vecs = updated_tfidv.transform(sentences)
    similarities = cosine_similarity(query_vec, sent_vecs).flatten()
    best_idx = np.argmax(similarities)
    answer = sentences[best_idx] + "."
    return answer, sources

demo_queries = [
    "Как измерить семантическую близость текстов?",
    "Что такое RAG и зачем он нужен?",
    "Какие метрики использовать для оценки поиска?"
]

print("Примеры работы mini-RAG:\n")
for q in demo_queries:
    ans, src = extractive_answer(q, top_k=2)
    print(f"Вопрос: {q}")
    print(f"Ответ: {ans}")
    print(f"Источники: {', '.join(src)}\n")

test_rag_queries = demo_queries + [
    "Как работает FAISS?",
    "Что такое overlap при чанкинге?",
    "Как обновить базу знаний?"
]

rag_examples = []
for q in test_rag_queries:
    ans, src = extractive_answer(q, top_k=2)
    rag_examples.append({
        'question': q,
        'answer': ans,
        'retrieved_sources': ', '.join(src)
    })

df_rag = pd.DataFrame(rag_examples)
os.makedirs('artifacts', exist_ok=True)
df_rag.to_csv('artifacts/rag_examples.csv', index=False, encoding='utf-8-sig')
print("Примеры сохранены в artifacts/rag_examples.csv")

Примеры работы mini-RAG:

Вопрос: Как измерить семантическую близость текстов?
Ответ: Эмбеддинги текстов применяются в задачах семантического поиска, кластеризации, классификации и перефразирования.
Источники: Эмбеддинги текстов

Вопрос: Что такое RAG и зачем он нужен?
Ответ: Также растёт интерес к RAG с потоковым обновлением знаний (real‑time RAG).
Источники: Будущее RAG и тенденции развития, Retrieval-Augmented Generation (RAG)

Вопрос: Какие метрики использовать для оценки поиска?
Ответ: Эти метрики позволяют объективно сравнивать разные стратегии поиска.
Источники: Метрики оценки качества retrieval: hit, recall, MRR, Косинусное сходство и его применение

Примеры сохранены в artifacts/rag_examples.csv


In [29]:
error_queries = [
    "Какая сегодня погода?",               
    "Что такое индекс в базе данных SQL?",  
    "Как нормализовать эмбеддинги?"         
]

print("Анализ ошибок:\n")
for q in error_queries:
    ans, src = extractive_answer(q, top_k=2)
    print(f"Запрос: {q}")
    print(f"Ответ: {ans}")
    print(f"Источники: {src}\n")
    if "погода" in q:
        print("  -> Проблема: запрос вне предметной области. Retrieval возвращает случайные чанки, ответ бессмыслен.")
    elif "SQL" in q:
        print("  -> Проблема: в базе знаний нет информации об индексах SQL. Retrieval находит ближайший по смыслу чанк про FAISS, но ответ неверен.")
    elif "нормализовать" in q:
        print("  -> Проблема: ответ есть в документе 'Эмбеддинги текстов', но экстрактивный генератор выбрал не то предложение из-за шума в контексте.")
    print("---")

print("Общие выводы:")
print("- Внепредметные запросы приводят к случайным ответам (нужна фильтрация).")
print("- Если база знаний не покрывает тему, RAG не может дать верный ответ.")
print("- Экстрактивный генератор сильно зависит от формулировки и выбора предложения.")
print("- Улучшение retrieval (реранжирование) и замена генератора на LLM могли бы исправить многие ошибки.")

Анализ ошибок:

Запрос: Какая сегодня погода?
Ответ: TF‑IDF (Term Frequency — Inverse Document Frequency) — это классический метод векторизации текстов, не использующий нейронные сети.
Источники: ['TF‑IDF как альтернатива эмбеддингам']

  -> Проблема: запрос вне предметной области. Retrieval возвращает случайные чанки, ответ бессмыслен.
---
Запрос: Что такое индекс в базе данных SQL?
Ответ: FAISS зависит от объёма данных, требуемой задержки и допустимой потери точности.
Источники: ['Индексирование в FAISS: типы индексов']

  -> Проблема: в базе знаний нет информации об индексах SQL. Retrieval находит ближайший по смыслу чанк про FAISS, но ответ неверен.
---
Запрос: Как нормализовать эмбеддинги?
Ответ: Эмбеддинги можно нормализовать, чтобы длина вектора стала равной единице.
Источники: ['Эмбеддинги текстов', 'Векторизация текстов: практические аспекты']

  -> Проблема: ответ есть в документе 'Эмбеддинги текстов', но экстрактивный генератор выбрал не то предложение из-за шума в контексте